# Whisper Fine-Tuning — Duration Sweep
--------

This notebook runs a series of fine-tuning experiments on progressively different datasets as shown in the table below:


| Section | Dataset | Notes |
|---------|---------|-------|
| A | Full 14h (no filtering) | Baseline run | 



## 1. Python Setup

In [1]:
ENV = "local"  # options: "local", "colab"

In [4]:
import os
import time
from datetime import datetime
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import login
import wandb

# Project path   
if ENV == "jupyter-hub":
    PROJECT_SRC = Path('/home/jupyter-wb344850/chichewa-asr')
else:
    # Modify index at the end as required to point to the parent directory of `src/` (which contains the `src` package)
    PROJECT_SRC = Path().cwd().parents[1]

sys.path.append(str(PROJECT_SRC))

from src.train.train_whisper import load_config
from src.train.whisper_duration_experiment import (
    load_model_and_processor,
    prepare_train_dataset,
    prepare_test_dataset,
    run_training,
    run_evaluation,
)

## 2. Paths and Global Configuration

In [9]:
# ==========================================
# 1. BASE WORKING DIRECTORY AND PATHS
# ==========================================
# Base directory of the project (one level up from src/)
if ENV == "jupyter-hub":
    DIR_BASE = Path('/home/jupyter-wb344850/chichewa-asr')
else:
    DIR_BASE = Path().cwd().parents[1]

DIR_DATA   = DIR_BASE / "data"
# Hold out test set 
DIR_TEST               = DIR_DATA / "test"
FILE_MANIFEST_TEST     = DIR_TEST / "metadata.csv"

# Audio files for the dev set and the dev set with nested duration
DIR_DEV                     = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"
DIR_DEV_NESTED_DURATION     = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE / "configs" / "whisper_hparams_baseline.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "datasets_experiments"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

# Model checkpoint directory (for saving model checkpoints during training)
DIR_MODEL_CHECKPOINTS = DIR_BASE / "models/checkpoints"
DIR_MODEL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. HARDWARE SETUP
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
# Human-readable label for the machine/device used — update as needed
DEVICE_LABEL = "mps-96gb-ram"  

Using device: cpu


In [6]:
# ==========================================
# 3. LOGIN TO HUGGINGFACE HUB AND WANDB
# ==========================================
# Load .env file
load_dotenv()

# Log into Huggingface Hub using token from .env file
login(token=os.getenv("HF_TOKEN"))

# Lof into Weights & Biases using API key from .env file
wandb.login(key=os.getenv("WANDB_API_KEY"))



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/dmatekenya/.netrc
wandb: Network error (SSLError), entering retry loop.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

## 4. Model Configurations and Held-Out Test Set Preparation 

These two objects are created once and shared across every duration run.
The test set is preprocessed with a temporary processor (base model weights);
audio features are model-agnostic so this is correct for all runs.

In [10]:
# ============================
# 1. LOAD BASE CONFIG ONCE
# ============================
base_config = load_config(FILE_CONFIG)
print("Config loaded:", FILE_CONFIG)

# =========================================
# 2. MODEL AND HUB SETTINGS
# =========================================
MODEL_ID     = base_config["model"]["model_name_or_path"]  
MODEL_NAME   = MODEL_ID.split("/")[-1]                     
BASE_HUB_MODEL_ID = f"dmatekenya/{MODEL_NAME}-chichewa"   

# =========================================
# 3. PREPARE HELD-OUT TEST SET
# =========================================
dataset_test = prepare_test_dataset(
    FILE_MANIFEST_TEST,
    audio_dir=DIR_TEST,
    base_config=base_config,
    audio_fname_col="audio_filename",
    duration_col="duration_seconds",
)

print(f"\nHeld-out test set ready: {len(dataset_test):,} utterances")


Config loaded: /Users/dmatekenya/git-repos/chichewa-asr/configs/whisper_hparams_baseline.yaml
  Loading test data: /Users/dmatekenya/git-repos/chichewa-asr/data/test/metadata.csv
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


Map (num_proc=1): 100%|██████████| 573/573 [00:13<00:00, 42.66 examples/s]


Held-out test set ready: 573 utterances


## 5. Run Experiments

### 5.1 Baseline Dataset
We run on the whole 14 hours without any filters 

In [ ]:
# ==========================================
# 1. MODEL NAME AND HUB SETTINGS
# ==========================================    
# Set duration label for this run
duration_label = "14h"
# Hub model ID for this run and output dir for this run 
hub_model_id = f"{BASE_HUB_MODEL_ID}-{duration_label}"
output_dir = DIR_MODEL_CHECKPOINTS / f"{MODEL_NAME}-chichewa-{duration_label}"

# ==========================================
# 2. LOAD MODEL AND PROCESSOR
# ==========================================  
model, processor = load_model_and_processor(base_config)

# ==========================================
# 3. PREPARE TRAINING DATA
# ========================================== 
dataset_train = prepare_train_dataset(FILE_MANIFEST_DEV, DIR_DEV, processor)

# ==========================================
# 4. TRAIN MODEL
# ==========================================
print(f"\n{'='*60}\n  EXPERIMENT: {duration_label}\n{'='*60}")
train_start = time.time()
trainer = run_training(model, processor, dataset_train, base_config, hub_model_id, output_dir)
train_minutes = (time.time() - train_start) / 60

# ==========================================
# 5. PUSH TO HUB (OPTIONAL)
# ==========================================
print(f"  Pushing to Hub: {hub_model_id}")
trainer.push_to_hub()

# ==========================================
# 6. EVALUATE ON HELD-OUT TEST SET
# ==========================================
row = run_evaluation(
    model,
    processor,
    dataset_test,
    duration_label,
    DIR_RESULTS,
    model_id=hub_model_id,
    debug=True,
)

single_run_result = {
    "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "duration": duration_label,
    "wer": row["wer"],
    "cer": row["cer"],
    "hub_model_id": hub_model_id,
    "device": DEVICE_LABEL,
    "train_minutes": round(train_minutes, 2),
}

pd.DataFrame([single_run_result]).to_csv(
    DIR_RESULTS / "duration_sweep_summary.csv",
    index=False
)

## 7. Inspect Summary Results 

In [ ]:
df_summary = pd.DataFrame(summary)
print("\nFinal summary of duration sweep:")
display(df_summary.style.format({"wer": "{:.2f}%", "cer": "{:.2f}%"}))
